??????? 6. ???????? ?????????
===========

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.style.use('ggplot')
%matplotlib inline

In [ ]:
import os
os.chdir("c:/aaa/temp_py/Shad_Python_08_2/")

### ???????????? ???? ????? ? ??????????

In [ ]:
df = pd.read_csv('Albuquerque Home Prices.txt', sep='\t')
df = df.replace(-9999, np.nan)

In [ ]:
df.head()

??????, ??? ??????? ??????, ????? ?????? ?????? ? ???????????? ??????????. ??????? ?????????, ??????? ??????? ?? ????????.

In [ ]:
print 'Rows in the data frame: {0}'.format(len(df))
print 'Rows without NAN: {0}'.format(len(df.dropna(how='any')))

????? ???????, ????? ???????? ????? ?????? ???????? ???? ?? ???? ??????????? ????????. ???????? ??, ??? ?????? ???????????? ????, ??????????? ??? ????? ???????, ???? ?? ??????? ?????????????.

????????? ??????? ?????? ? ?????????? ?? ????????????? ??????????? ???????? ?? ????????.

??????? ```DataFrame.apply``` ????????? ????????? ??????? ?? ???? ???????. ?????? ???????? -- ??????????? ???????, ?????? ???????? -- ??????????? ?? ?????????? (0 -- ????????? ? ????????, 1 -- ?? ????????).

In [ ]:
df.apply(lambda x: sum(x.isnull()), axis=0)

??? ??????????? ??????? ????? ???????????? ??? ?????????? ??????-????????? (??????-???????). ??? ?????? ???????? ???????? ??????? ? ???? ???????, ?? ??????? ?? ?????. ? ?????? ?????? ????? ??????-????????? ???????????? ?????????? ??????????? ???????:

`
def f(x):
    return sum(x.isnull())
`

????? ???????, ?????, ??? ??????????? ??????????? ???????? ???????? ??????? ```AGE``` ? ```TAX```. ??? ? ??????? ```AGE``` ??? ????? ?????????, ?? ????? ?? ????? ?????? ?? ???????, ??? ???????? ???????? ??? ?????.

In [ ]:
del df['AGE']

? ??????? ```TAX``` ????? ??????????? ????????. ????? ??? ???????, ??????? ?????????, ??? ???????? ????????????? ???? ???????.

In [ ]:
df['TAX'].hist()

??? ????????????? ?????? ?? ??????????, ?? ????? ???????? ???????? ?? ??????? ???????? ???? ???????. ??? ????? ? pandas ???? ??????? ```Series.fillna```, ??????? ???????? ??? ??????????? ? ??????? ???????? ?? ????????.

In [ ]:
df['TAX'] = df['TAX'].fillna(df['TAX'].mean())

????????, ??? ??? ???????? ? ?????????? ?????????. ????? ??????? ? ?????????? ????? ????? ????????? ????????.

In [ ]:
len(df.dropna())

#### ?????????? ??????

?????? ????? ?????????? ??????????????? ? ?????????? ??????.

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
X = df.drop('PRICE', axis=1)
y = df['PRICE']

In [ ]:
model = LinearRegression()
model.fit(X, y)

??????? ???????? ?????? (??????????? $R^2$).

In [ ]:
print 'R^2: {0}'.format(model.score(X, y))

??????? ????????????? ???????????? ?? ?????? ```model.coef_``` ? ????????? ???? ?? ?????? ```model.intercept_```.

In [ ]:
coef = pd.DataFrame(zip(['intercept'] + X.columns.tolist(), [model.intercept_] + model.coef_.tolist()),
                    columns=['predictor', 'coef'])
coef

? ?????????, ? ```sklearn``` ?? ????????????? ????????? ??????????? ?????????????? ?????????? ????????????? ?????????????. ??????? ????? ??? ?????? ???? ??????, ???????????? ??????? ? ????????????? ?????????????, ???? ???????????????? ??????? ?? ?????? ```statsmodels```.

In [ ]:
from scipy import stats

def regression_coef(model, X, y):
    coef = pd.DataFrame(zip(['intercept'] + X.columns.tolist(), [model.intercept_] + model.coef_.tolist()),
                    columns=['predictor', 'coef'])
    X1 = np.append(np.ones((len(X),1)), X, axis=1)
    b = np.append(model.intercept_, model.coef_)
    MSE = np.sum((model.predict(X) - y) ** 2, axis=0) / float(X.shape[0] - X.shape[1])
    var_b = MSE * (np.linalg.inv(np.dot(X1.T, X1)).diagonal())
    sd_b = np.sqrt(var_b)
    t = b / sd_b
    coef['pvalue'] = [2 * (1 - stats.t.cdf(np.abs(i), (len(X1) - 1))) for i in t]
    return coef
regression_coef(model, X, y)

??? ???????????????? ??????? ? ??? ???? 3 ???????? ??????????: ```SQFT```, ```CUST``` ? ```TAX```.  ???????? ???????????? ???????, ?? ??????? ???????? ???????? ??????? ??? ?????????? ?????????? ?? ???? ???????. ????? ?????? ???? ????????????? ???????: ??????? ? ??????????? ??? ????????? ???????? ?????? ?? ????. ?? ???? ?? ????? ???????? ? ???????????? ???? ???? ????? ? ????? ?? ????? ???? ?????, ??? ? ????, ????? ??????????? ?????? ?? ??????? ???????????. ??? ?????? ?????? ?????????????. ? ?????????????. ???????? ??????? ?????? ?????????? ???????????.

In [ ]:
df.corr()

?????? ?????? ??????????? ? ????????. ???????????? ???????? ??????.

In [ ]:
X = df[['SQFT', 'CUST']]
model.fit(X, y)
print model.score(X, y)
regression_coef(model, X, y)

### ?????????????? ?????????

In [ ]:
df = pd.read_csv('diamond.dat', header=None, sep='\s+', names=['weight', 'price'])

In [ ]:
df.head()

????? ????????????? ???? ?????? ? ??????????? ?? ???? ??????????. ????????? ??????????? ????????? ?? ?????? ? ??????? ????? ??????????, ?? ? ? ??????? ?? ????????. ?????? ?????? ????? ???????? ?????? ?? ????? ?????????? ```weight```, ?????? -- ?? ?????????? ```weight^2```, ? ?????? -- ?? ???? ???? ?????????? ?????.

? ???? ???????, ????? ?? ????? ??????????? ? ???????? ????? ?? ??????? ? ???????????? ???, ? ```sklearn.preprocessing``` ???? ????? ```PolynomialFeatures```. ????? ```fit_transform``` ????? ?????? ??????????? ?? ??????? ????????? ??? ????????? ?????????? ???????? ???????. ????????, ??? ??????? 2 ? ??? ```a```, ```b``` ????? ????????????? ???? ```[a, b, a^2, b^2, ab]```, ? ??? ????????? ????????? ```include_bias=True``` ??? ? ??????-????????? ???? ?? ??????. ??? ?????? ??????, ???????, ????? ???? ????????? ???? ??????? ?? ????????? ???????? ??????? ```weight```. ?? ??? ????? ???????????? ? ??????? ```PolynomialFeatures``` ??? ??????????? ??????????.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(
                          # ???????????? ???????
                          degree=2,
                          # ?? ???????????? ????????? ????
                          include_bias=False)
y = df['price']
X0 = poly.fit_transform(df[['weight']])
X0 = pd.DataFrame(X0, columns=['weight', 'weight^2'])
X0.head()

In [ ]:
X0 = [
    # ???? ???????????? ?????????? weight
    X0[['weight']],
    # ???? ?????????? weight^2
    X0[['weight^2']],
    # ??? ?????????? weight ? weight^2
    X0.copy()]
models = [LinearRegression() for _ in X0]

In [ ]:
for X, model in zip(X0, models):
    model.fit(X, y)
    print model.score(X, y)

??????????, ??? $R^2$ ?? ???? ??????? ???????? ????????. ?????? ?? ??? ??? ????????? ???????. 

In [ ]:
regression_coef(models[0], X0[0], y)

In [ ]:
regression_coef(models[1], X0[1], y)

In [ ]:
regression_coef(models[2], X0[2], y)

In [ ]:
import statsmodels.api as sm
X2 = sm.add_constant(X0[2])
est = sm.OLS(y, X2)
est2 = est.fit()
print(est2.summary())

### ??????????????? ????????? ?????.

In [ ]:
df = pd.read_csv('series_g.csv', sep=';')
df.head()

In [ ]:
# ??????????? ??????? ? ?????? ? ?????? datetime
# format='%b %Y' ????????, ??? ? ????? ??????? ??????? ???? ????????????? ???????? ?????? (%b), ? ????? ??? (%Y)
df['date'] = pd.to_datetime(df['date'], format='%b %Y')

???????? ??????? ?????? ?????????????????? ? ??????? ?? 4 ???????.
1 ???? ?? ????? ? ?????????? ????, ? ???? ????, ?? ??????
2 ???? ?? ?????????? ? ?????????? ????, ? ???? ????, ?? ????? ( ?????????? ??? ?????????????????). 
3 ?????? ?? ??? ???? ?????????
4 ???? ?? ????????
?? ??????? ??????? ??? ?????????????? ???????? ????? ? ????????????????? ??????????. 
????? ???????? ?????: ???????????????? ???????? ????. 
????? ???????????????? ? ????? ?????? ?????????? ????? ?????????? ??????, ? ??? ? ??????? ?? ?????????? ????????? ??????????.

In [ ]:
fig = plt.figure(figsize=(12, 4))
ax1 = fig.add_subplot(121)
df['series_g'].plot(ax=ax1)
ax1.set_title(u'????? ??????????????????')
ax1.set_ylabel(u'????? ???????')

ax2 = fig.add_subplot(122)
pd.Series(np.log10(df['series_g'])).plot(ax=ax2)
ax2.set_title(u'log10 ?? ?????? ??????????????????')
ax2.set_ylabel(u'log10 ?? ????? ???????')


?????: ????? ??????? ?????? ???????? ????????? ??? ??????????? ????????? ?? ?????? ?????????. ?? ????
$$\log y_i = \beta x_i + c(x_i) + \varepsilon_i,$$
??? $y_i$ -- ????? ?????????, $x_i$ -- ?????????? ????? ??????, $c(x_i)$ -- ???????? ????????????, $\varepsilon_i$ -- ????????? ???.

??? ???????? ??????????? ????????????? ???????? ????????????? 12 ????? ??????? ??? ?????????? ???????? ?? ???. ??? ????? ???????? ??? 12 ????? ??? ? ??????? ??????? ```pd.date_range```. ?????? ?????? ????? ???????? ?????? ```DateTimeIndex``` (????????????? ?? ?????? ```Index```), ? ????? ?????????? ?? ? ???????? ```df['date']```, ????????????? ?????? ```datetime64```, ???????? ???????? ????????? ? ?????? ```Index```. ????????? ??? ?????? ??? ? ???????? ?? ? ??????? ```new_dates```.

????? ???????? ????????? ????????? ```df2```, ????????? ?? ????? ??????? ? ????? ?????? ??????, ? ???????? ??? ? ????????? ?????????? ```df``` ? ??????? ??????? ```pd.merge```. ??? ??????? ????????? ??? ???????? ?? ?????????? ?????? ??????? (???????? ```on```) ? ?? ?????????? ??????? ??????? (???????? ```how```). ? ```on``` ????????? ???? ????? ??????? ```date```, ?? ??????? ????? ?????????? ???????. ? ```how``` ????????? ```right```, ??? ???????? ?????????: 
?????? ???? ?????? ??????? ? ??????? ? ???? ????? ??????? ?? ??????? ?????????? ???????? ??????? ```on```, ? ? ?????? ???? ??? ???????? ?? ?????? ??????? ```on``` ?? ???????? ?????????????? ???????? ? ????? ??????? ```on```, ?? ????? ??????? ```NaN``` ????????. ?????? ??????, ????? ```how``` ????????????? ?????? ```JOIN``` ? ????? SQL (```LEFT JOIN```, ```RIGHT_JOIN```, ```INNER JOIN```, ```OUTER_JOIN```).

In [ ]:
# ??????? ???????????????? ???????. freq='MS' ???????? ?????? ????? ??????? ?????? ?? ?????????? ????????? ???
new_dates = pd.date_range('1961-01-01', '1961-12-01', freq='MS')
# ???????? df['date'] ? ???? Index, ?????????? ? 12 ????????, ??????????? ?? ?????????? ????
new_dates = pd.Index(df['date']) | new_dates
# ??????? ????????? ?? ????? ??????? ? ??????????? ??????? ???
df2 = pd.DataFrame({'date': new_dates})
# ?????????? ??? ?????????? ?? ??????? 'date'.
df = pd.merge(df, df2, on='date', how='right')

???????? ????????????? ?????????? ```month_num``` -- ?????????? ????? ???? (?????, ???). ? ???????????????? ??????.

In [ ]:
df['month_num'] = range(1, len(df) + 1)
df['log_y'] = np.log10(df['series_g'])

???????? 12 ??????? ```season_1```, ```season_2```, ..., ```season_12```, ? ??????? ???????? ?????????? ???????????????? ??????. ????? ??????? ?????????? ????? ?????? ? ?????? ???????, ???????? ??????????????? ???? ??????? ```dt``` ? ```month``` ? ??????? ```df['date']```. ?????? ????? ????? ?????????, ????? ?? ????????? ????? ???????? ???????? ?? ?????.

In [ ]:
for x in range(1, 13):
    df['season_' + str(x)] = df['date'].dt.month == x

??????, ??? ?????????? ???????? ??????????? ????? ?????????, ???? ?? ???????? ??????????? ???????? ?????????. ????? ??????? ??????? ????? ??????.

In [ ]:
# xrange(2, 13) ????????????? ???? ??????? ? ??????? ?? ???????
season_columns = ['season_' + str(x) for x in range(2, 13)]

# ???????? ??????? ??????? X ? ?????? y ??? ???????? ??????
X = df[['month_num'] + season_columns]
y = df['log_y']

# ??????? ?????? ?? ???????, ? ??????? ???????? ???????? y (? ??????? < 144)
X1 = X[X.index < 144]
y1 = y[y.index < 144]

??????????? ???????? ????????????? ??????.

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
model = LinearRegression()
model.fit(X1, y1)

In [ ]:
pred = pd.DataFrame({
    'pred': model.predict(X1),
    'real': y1})
pred.plot()

?????? ???????? ???????????? ??? ???? ??????? ```X```, ??????? ??????????? 12 ???????.

In [ ]:
pred = pd.DataFrame({
    'pred': model.predict(X),
    'real': y})
pred.plot()